In [3]:
import string
from collections import Counter

alfabeto = string.ascii_uppercase

# limpiar texto
def limpiar(texto):
    return "".join([c for c in texto if c in alfabeto])

# índice de coincidencia
def indice_coincidencia(texto):
    N = len(texto)
    frec = Counter(texto)
    ic = sum(f*(f-1) for f in frec.values()) / (N*(N-1)) if N > 1 else 0
    return ic

# encontrar longitud probable de clave
def encontrar_longitud(texto, max_len=10):
    mejores = []
    for n in range(1, max_len+1):
        bloques = ['' for _ in range(n)]
        for i, c in enumerate(texto):
            bloques[i % n] += c

        ic_medio = sum(indice_coincidencia(b) for b in bloques) / n
        mejores.append((n, ic_medio))

    mejores.sort(key=lambda x: x[1], reverse=True)
    return mejores[0][0]  # mejor longitud


# descifrar césar por frecuencia (español aproximado)
def descifrar_cesar(bloque):
    frecuencias_es = "EAOSRNIDLCTUMPBGYVQHFZJÑXKW"  # orden típico español
    mejor = ""
    mejor_score = -1
    mejor_k = 0

    for k in range(26):
        resultado = ""
        for c in bloque:
            nueva = chr((ord(c) - ord('A') - k) % 26 + ord('A'))
            resultado += nueva

        score = sum(resultado.count(letra) for letra in "EAO")

        if score > mejor_score:
            mejor_score = score
            mejor = resultado
            mejor_k = k

    return mejor, mejor_k


# ataque completo
def ataque_vigenere(texto):
    texto = limpiar(texto)

    longitud = encontrar_longitud(texto)
    print(f"🔑 Longitud probable de clave: {longitud}")

    bloques = ['' for _ in range(longitud)]
    for i, c in enumerate(texto):
        bloques[i % longitud] += c

    clave = ""
    descifrados = []

    for b in bloques:
        desc, k = descifrar_cesar(b)
        descifrados.append(desc)
        clave += chr(k + ord('A'))

    # reconstruir texto
    resultado = [""] * len(texto)
    for i, bloque in enumerate(descifrados):
        for j, c in enumerate(bloque):
            resultado[j * longitud + i] = c

    return clave, "".join(resultado)


# --- EJECUCIÓN ---
with open("texto_cifrado.txt", "r") as f:
    texto = f.read()

clave, descifrado = ataque_vigenere(texto)

print("\n🔑 Clave encontrada:", clave)
print("\n📜 Texto descifrado (inicio):\n")
print(descifrado[:500])

🔑 Longitud probable de clave: 8

🔑 Clave encontrada: PASSPASS

📜 Texto descifrado (inicio):

ALREDEDORDELACATEDRALSEEXTENDIAENESTRECHAZONAELPRIMITIVORECINTODEVETUSTACOMPRENDIALOQUESELLAMABAELBARRIODELAENCIMADAYDOMINABATODOELPUEBLOQUESEHABIAIDOESTIRANDOPORNOROESTEYPORSUDESTEDESDELATORRESEVEIAENALGUNOSPATIOSYJARDINESDECASASVIEJASYRUINOSASRESTOSDELAANTIGUAMURALLACONVERTIDOSENTERRADOSOPAREDESMEDIANERASENTREHUERTOSYCORRALESLAENCIMADAERAELBARRIONOBLEYELBARRIOPOBREDEVETUSTALOSMASLINAJUDOSYLOSMASANDRAJOSOSVIVIANALLICERCAUNOSDEOTROSAQUELLOSASUSANCHASLOSOTROSAPINADOSELBUENVETUSTENTEERADELAENCIMAD
